# DataFrame API Basics

{doc}`Introduction to DataFrames <../01_introduction_to_apache_spark/02_spark_dataframes_and_sql/01_introduction_to_dataframes>`
introduced the DataFrame, the difference between transformations and actions,
and the handful of methods you need to get started. This notebook builds on
that foundation with the everyday tools of real application code: inspecting
an unfamiliar table, deriving many columns at once, casting safely, writing
filters that do what you mean, conditional logic, strings and timestamps,
missing values, and removing duplicates.

All notebooks in this chapter run against Databricks serverless compute
through Databricks Connect, as set up in
{doc}`Setting Up Databricks Connect <../00_getting_started/01_setting_up_databricks_connect>`.

## The Bakehouse Dataset

Every Databricks workspace ships with read-only sample data in the `samples`
catalog. This chapter uses `samples.bakehouse`, the sales records of a
fictional chain of bakery franchises:

| Table | Grain | Key columns |
| --- | --- | --- |
| `sales_transactions` | One row per sale | `transactionID`, `customerID`, `franchiseID` |
| `sales_customers` | One row per customer | `customerID` |
| `sales_franchises` | One row per bakery | `franchiseID`, `supplierID` |
| `sales_suppliers` | One row per ingredient supplier | `supplierID` |
| `media_customer_reviews` | One row per online review | `franchiseID` |

The tables are small enough that every query here returns in a few seconds,
but every technique on this page works unchanged on billions of rows.

Start a session and read the transactions table:

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

transactions = spark.read.table("samples.bakehouse.sales_transactions")

## Inspecting a DataFrame

When you meet a new table, the first questions are what columns it has,
what types they are, and what the values look like. `printSchema` answers
the first two:

In [2]:
transactions.printSchema()

root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



The same information is available programmatically through `columns`,
`dtypes` and `schema`, which is useful when code needs to adapt to the
columns it is given:

In [3]:
transactions.dtypes

[('transactionID', 'bigint'),
 ('customerID', 'bigint'),
 ('franchiseID', 'bigint'),
 ('dateTime', 'timestamp'),
 ('product', 'string'),
 ('quantity', 'bigint'),
 ('unitPrice', 'bigint'),
 ('totalPrice', 'bigint'),
 ('paymentMethod', 'string'),
 ('cardNumber', 'bigint')]

`show` prints rows. Two options make it far more readable on wide tables:
`truncate=False` stops long values from being cut off at 20 characters, and
`vertical=True` prints one block per row instead of one very wide line.

In [4]:
transactions.show(2, vertical=True)

-RECORD 0-----------------------------
 transactionID | 1002961              
 customerID    | 2000253              
 franchiseID   | 3000047              
 dateTime      | 2024-05-14 12:17:... 
 product       | Golden Gate Ginger   
 quantity      | 8                    
 unitPrice     | 3                    
 totalPrice    | 24                   
 paymentMethod | amex                 
 cardNumber    | 378154478982993      
-RECORD 1-----------------------------
 transactionID | 1003007              
 customerID    | 2000226              
 franchiseID   | 3000047              
 dateTime      | 2024-05-10 23:10:... 
 product       | Austin Almond Bis... 
 quantity      | 36                   
 unitPrice     | 3                    
 totalPrice    | 108                  
 paymentMethod | mastercard           
 cardNumber    | 2244626981238094     
only showing top 2 rows


To get a feel for the *values*, `describe` computes count, mean, standard
deviation, minimum and maximum for every column you pass it. `summary` does
the same and adds quartiles:

In [5]:
transactions.select("quantity", "unitPrice", "totalPrice").summary().show()

+-------+-----------------+---------+------------------+
|summary|         quantity|unitPrice|        totalPrice|
+-------+-----------------+---------+------------------+
|  count|             3333|     3333|              3333|
|   mean|6.647764776477648|      3.0|19.943294329432945|
| stddev|6.544484308501296|      0.0|  19.6334529255051|
|    min|                1|        3|                 3|
|    25%|                3|        3|                 9|
|    50%|                6|        3|                18|
|    75%|                9|        3|                27|
|    max|               60|        3|               180|
+-------+-----------------+---------+------------------+



Every transaction sells a product at a unit price of 3, quantities range from
1 to 60, and the quartiles show that half of all sales are between
3 and 9 items.

:::{admonition} Note: `show`, `describe` and `summary` Are Actions
:class: note

Each of these runs a Spark job. That is fine while exploring, but in a
pipeline every stray `show` re-runs the plan, as explained in
{doc}`Introduction to DataFrames <../01_introduction_to_apache_spark/02_spark_dataframes_and_sql/01_introduction_to_dataframes>`.
Remove them, or guard them behind a debug flag, before the code goes to
production.
:::

## Deriving Columns

`withColumn` adds or replaces a single column. When you derive several
columns, `withColumns` takes a dictionary and adds them all in one step,
which keeps the query plan small (see
{ref}`Driver Code versus Executor Code <driver-vs-executor-code>` for why
that matters):

In [6]:
enriched = transactions.withColumns(
    {
        "sale_date": F.to_date("dateTime"),
        "sale_hour": F.hour("dateTime"),
        "weekday": F.date_format("dateTime", "EEEE"),
        "is_bulk_order": F.col("quantity") >= 30,
    }
)

enriched.select("transactionID", "dateTime", "sale_date", "sale_hour", "weekday", "is_bulk_order").show(5, truncate=False)

+-------------+--------------------------+----------+---------+--------+-------------+
|transactionID|dateTime                  |sale_date |sale_hour|weekday |is_bulk_order|
+-------------+--------------------------+----------+---------+--------+-------------+
|1002961      |2024-05-14 12:17:01.495952|2024-05-14|12       |Tuesday |false        |
|1003007      |2024-05-10 23:10:10.239954|2024-05-10|23       |Friday  |true         |
|1003017      |2024-05-16 16:34:10.61372 |2024-05-16|16       |Thursday|true         |
|1003068      |2024-05-02 04:31:51.612094|2024-05-02|4        |Thursday|false        |
|1003103      |2024-05-04 23:44:26.902224|2024-05-04|23       |Saturday|false        |
+-------------+--------------------------+----------+---------+--------+-------------+
only showing top 5 rows


Its siblings `withColumnsRenamed` and `drop` rename and remove several
columns at once. Here we tidy the names into `snake_case` and drop the card
number, which nothing downstream should ever need:

In [7]:
tidy = enriched.withColumnsRenamed(
    {
        "transactionID": "transaction_id",
        "customerID": "customer_id",
        "franchiseID": "franchise_id",
        "unitPrice": "unit_price",
        "totalPrice": "total_price",
        "paymentMethod": "payment_method",
    }
).drop("cardNumber", "dateTime")

tidy.columns

['transaction_id',
 'customer_id',
 'franchise_id',
 'product',
 'quantity',
 'unit_price',
 'total_price',
 'payment_method',
 'sale_date',
 'sale_hour',
 'weekday',
 'is_bulk_order']

`F.lit` turns a Python value into a column, for when every row should get
the same value:

In [8]:
tidy.select("transaction_id", F.lit("USD").alias("currency")).show(3)

+--------------+--------+
|transaction_id|currency|
+--------------+--------+
|       1002961|     USD|
|       1003007|     USD|
|       1003017|     USD|
+--------------+--------+
only showing top 3 rows


## Data Types and Casting

`cast` converts a column to another type. You can name the target type with
a DDL string such as `"int"`, `"decimal(10,2)"` or `"date"`, or with a type
object from `pyspark.sql.types`:

In [9]:
tidy.select(
    "total_price",
    F.col("total_price").cast("decimal(10,2)").alias("as_decimal"),
    F.col("total_price").cast("string").alias("as_string"),
).show(3)

+-----------+----------+---------+
|total_price|as_decimal|as_string|
+-----------+----------+---------+
|         24|     24.00|       24|
|        108|    108.00|      108|
|        120|    120.00|      120|
+-----------+----------+---------+
only showing top 3 rows


Spark 4 runs in **ANSI mode** by default, and so does Databricks serverless
compute. In ANSI mode an invalid cast is an error, not a silent `NULL`:

In [10]:
spark.conf.get("spark.sql.ansi.enabled")

'true'

In [11]:
raw_quantities = spark.createDataFrame(
    [("12",), ("7",), ("a dozen",)],
    schema="quantity STRING",
)

try:
    raw_quantities.select(F.col("quantity").cast("int")).show()
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0])

NumberFormatException - [CAST_INVALID_INPUT] The value 'a dozen' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018


When bad values are expected and should become `NULL` instead, use
`try_cast`. The `try_` prefix marks the forgiving variant of several
functions, such as `try_divide`, `try_add` and `try_to_timestamp`:

In [12]:
raw_quantities.select("quantity", F.col("quantity").try_cast("int").alias("parsed")).show()

+--------+------+
|quantity|parsed|
+--------+------+
|      12|    12|
|       7|     7|
| a dozen|  NULL|
+--------+------+



## Filtering Rows

`filter` (and its alias `where`) keeps the rows for which a boolean column
expression is true. Besides the comparison operators, `Column` has methods
for the most common tests:

| Expression | Keeps rows where |
| --- | --- |
| `F.col("x").isin("a", "b")` | `x` is one of the listed values |
| `F.col("x").between(1, 10)` | `1 <= x <= 10` (both ends inclusive) |
| `F.col("x").like("Golden%")` | `x` matches a SQL `LIKE` pattern |
| `F.col("x").rlike("^[A-Z]")` | `x` matches a regular expression |
| `F.col("x").startswith("Go")` | `x` starts with a prefix |
| `F.col("x").isNull()` / `isNotNull()` | `x` is, or is not, missing |

Conditions are combined with `&` (and), `|` (or) and `~` (not), **not** with
Python's `and`, `or` and `not`. Because `&` and `|` bind more tightly than
comparisons in Python, every comparison must be wrapped in parentheses:

In [13]:
large_card_sales = tidy.filter(
    (F.col("quantity") >= 30)
    & F.col("payment_method").isin("visa", "mastercard")
    & ~F.col("product").startswith("Golden")
)

large_card_sales.count()

33

Forgetting the parentheses does not give you a wrong answer. It fails
outright, because Python evaluates `30 & F.col(...)` first and then tries to
turn a `Column` into a Python `bool`:

In [14]:
try:
    tidy.filter(F.col("quantity") >= 30 & F.col("payment_method") == "visa")
except Exception as e:
    print(type(e).__name__, "-", e)

PySparkValueError - [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.


A filter can also be a SQL string, which is often shorter for simple
conditions:

In [15]:
tidy.filter("quantity >= 30 AND payment_method IN ('visa', 'mastercard')").count()

40

## Conditional Logic with `when`

`F.when(condition, value)` is the DataFrame version of SQL's `CASE WHEN`.
Chain further `when` calls for more branches and finish with `otherwise`;
the first condition that matches wins. Without `otherwise`, unmatched rows
get `NULL`.

In [16]:
sized = tidy.withColumn(
    "order_size",
    F.when(F.col("quantity") >= 30, "large")
    .when(F.col("quantity") >= 10, "medium")
    .otherwise("small"),
)

sized.select("transaction_id", "quantity", "order_size").show(5)

+--------------+--------+----------+
|transaction_id|quantity|order_size|
+--------------+--------+----------+
|       1002961|       8|     small|
|       1003007|      36|     large|
|       1003017|      40|     large|
|       1003068|      28|    medium|
|       1003103|      28|    medium|
+--------------+--------+----------+
only showing top 5 rows


:::::{dropdown} Example: How Many Orders Are Large?
:open:
:color: primary
:icon: pencil
:class-body: example-body

Using the `order_size` column from above, what share of the 3,333
transactions are large orders of 30 items or more?

**Count the large orders.**

```python
large = sized.filter(F.col("order_size") == "large").count()
total = sized.count()
```

**Divide.**

$$
\frac{\text{large orders}}{\text{all orders}} = \frac{66}{3333} \approx 0.020
$$

Only about 2% of sales are large orders. Most customers buy a handful of
items: the median quantity, from the `summary` output above, is 6.
:::::

Checking the hand calculation against Spark:

In [17]:
large = sized.filter(F.col("order_size") == "large").count()
total = sized.count()

large, total, round(large / total, 3)

(66, 3333, 0.02)

## Strings and Timestamps

`pyspark.sql.functions` contains a large library of string and date
functions. A few of the most useful ones, applied to the bakehouse data:

In [18]:
tidy.select(
    "product",
    F.upper("product").alias("upper"),
    F.regexp_replace("product", r"\s+", "_").alias("slug"),
    F.split("product", " ").getItem(0).alias("first_word"),
    F.concat_ws(" / ", "product", "payment_method").alias("label"),
).show(5, truncate=False)

+----------------------+----------------------+----------------------+----------+-----------------------------------+
|product               |upper                 |slug                  |first_word|label                              |
+----------------------+----------------------+----------------------+----------+-----------------------------------+
|Golden Gate Ginger    |GOLDEN GATE GINGER    |Golden_Gate_Ginger    |Golden    |Golden Gate Ginger / amex          |
|Austin Almond Biscotti|AUSTIN ALMOND BISCOTTI|Austin_Almond_Biscotti|Austin    |Austin Almond Biscotti / mastercard|
|Austin Almond Biscotti|AUSTIN ALMOND BISCOTTI|Austin_Almond_Biscotti|Austin    |Austin Almond Biscotti / mastercard|
|Pearly Pies           |PEARLY PIES           |Pearly_Pies           |Pearly    |Pearly Pies / amex                 |
|Pearly Pies           |PEARLY PIES           |Pearly_Pies           |Pearly    |Pearly Pies / visa                 |
+----------------------+----------------------+---------

For timestamps, `date_trunc` rounds down to a unit such as an hour or a
week, `date_add` and `datediff` do calendar arithmetic, and `date_format`
renders a timestamp with a
[datetime pattern](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html):

In [19]:
enriched.select(
    "dateTime",
    F.date_trunc("hour", "dateTime").alias("hour_bucket"),
    F.date_add("sale_date", 7).alias("one_week_later"),
    F.datediff(F.lit("2024-06-01").cast("date"), "sale_date").alias("days_until_june"),
    F.date_format("dateTime", "yyyy-MM-dd HH:mm").alias("formatted"),
).show(5, truncate=False)

+--------------------------+-------------------+--------------+---------------+----------------+
|dateTime                  |hour_bucket        |one_week_later|days_until_june|formatted       |
+--------------------------+-------------------+--------------+---------------+----------------+
|2024-05-14 12:17:01.495952|2024-05-14 12:00:00|2024-05-21    |18             |2024-05-14 12:17|
|2024-05-10 23:10:10.239954|2024-05-10 23:00:00|2024-05-17    |22             |2024-05-10 23:10|
|2024-05-16 16:34:10.61372 |2024-05-16 16:00:00|2024-05-23    |16             |2024-05-16 16:34|
|2024-05-02 04:31:51.612094|2024-05-02 04:00:00|2024-05-09    |30             |2024-05-02 04:31|
|2024-05-04 23:44:26.902224|2024-05-04 23:00:00|2024-05-11    |28             |2024-05-04 23:44|
+--------------------------+-------------------+--------------+---------------+----------------+
only showing top 5 rows


:::{admonition} Note: Session Time Zone
:class: note

`TIMESTAMP` values are stored as instants in UTC and displayed in the
session time zone, `spark.sql.session.timeZone`. Functions like `hour` and
`to_date` also use it, so the same row can land on a different date for two
users with different session settings. Set the time zone explicitly in
pipelines whose results depend on calendar boundaries.
:::

## Missing Values

The bakehouse tables happen to have no missing values, so we build a small
DataFrame that does:

In [20]:
reviews = spark.createDataFrame(
    [
        ("Pearly Pies", 5, "Loved it"),
        ("Orchard Oasis", None, "No rating given"),
        ("Tokyo Tidbits", 3, None),
        (None, 4, "Forgot the product"),
    ],
    schema="product STRING, stars INT, comment STRING",
)

reviews.show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|  Pearly Pies|    5|          Loved it|
|Orchard Oasis| NULL|   No rating given|
|Tokyo Tidbits|    3|              NULL|
|         NULL|    4|Forgot the product|
+-------------+-----+------------------+



A `NULL` means "unknown", and Spark follows SQL's three-valued logic: any
comparison with `NULL` is itself `NULL`, which a filter treats as false.
This is a frequent source of silently lost rows. The filter below looks like
"everything except Pearly Pies" but also drops the row with no product:

In [21]:
reviews.filter(F.col("product") != "Pearly Pies").show()

+-------------+-----+---------------+
|      product|stars|        comment|
+-------------+-----+---------------+
|Orchard Oasis| NULL|No rating given|
|Tokyo Tidbits|    3|           NULL|
+-------------+-----+---------------+



To keep the unknown rows, say so explicitly:

In [22]:
reviews.filter((F.col("product") != "Pearly Pies") | F.col("product").isNull()).show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|Orchard Oasis| NULL|   No rating given|
|Tokyo Tidbits|    3|              NULL|
|         NULL|    4|Forgot the product|
+-------------+-----+------------------+



The `na` accessor replaces or removes missing values. `fillna` takes a
dictionary of per-column defaults, and `dropna` removes rows with missing
values, optionally only in the columns listed in `subset`:

In [23]:
reviews.fillna({"stars": 0, "comment": "(none)"}).show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|  Pearly Pies|    5|          Loved it|
|Orchard Oasis|    0|   No rating given|
|Tokyo Tidbits|    3|            (none)|
|         NULL|    4|Forgot the product|
+-------------+-----+------------------+



In [24]:
reviews.dropna(subset=["product", "stars"]).show()

+-------------+-----+--------+
|      product|stars| comment|
+-------------+-----+--------+
|  Pearly Pies|    5|Loved it|
|Tokyo Tidbits|    3|    NULL|
+-------------+-----+--------+



`F.coalesce` returns the first non-null value among its arguments, which is
handy for falling back from one column to another:

In [25]:
reviews.select("product", F.coalesce("comment", "product", F.lit("?")).alias("display_text")).show()

+-------------+------------------+
|      product|      display_text|
+-------------+------------------+
|  Pearly Pies|          Loved it|
|Orchard Oasis|   No rating given|
|Tokyo Tidbits|     Tokyo Tidbits|
|         NULL|Forgot the product|
+-------------+------------------+



## Sorting and Limiting

`orderBy` (alias `sort`) accepts column names or `Column` expressions.
`F.desc` reverses the order, and the `asc_nulls_last` family controls where
`NULL`s go:

In [26]:
reviews.orderBy(F.col("stars").desc_nulls_last(), "product").show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|  Pearly Pies|    5|          Loved it|
|         NULL|    4|Forgot the product|
|Tokyo Tidbits|    3|              NULL|
|Orchard Oasis| NULL|   No rating given|
+-------------+-----+------------------+



Sorting a whole DataFrame is a wide transformation that shuffles every row
(see
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`).
`orderBy(...).limit(n)` is cheap, though: Spark recognises the pattern and
keeps only the top *n* rows of each partition before combining them.

In [27]:
tidy.orderBy(F.desc("total_price")).limit(3).select("transaction_id", "product", "quantity", "total_price").show()

+--------------+--------------------+--------+-----------+
|transaction_id|             product|quantity|total_price|
+--------------+--------------------+--------+-----------+
|       1001320|Austin Almond Bis...|      60|        180|
|       1001783|         Pearly Pies|      60|        180|
|       1002065|     Outback Oatmeal|      60|        180|
+--------------+--------------------+--------+-----------+



## Removing Duplicates

`distinct` removes rows that are identical in every column. To find the
unique combinations of a few columns, select them first:

In [28]:
tidy.select("product", "payment_method").distinct().count()

18

`dropDuplicates` takes a list of columns and keeps one row per unique
combination of *those* columns, with all the other columns intact. Which
row survives is not defined. When it matters, use a window function as shown
in {doc}`ETL with the DataFrame API <../01_introduction_to_apache_spark/04_etl_with_the_dataframe_api/index>`.

In [29]:
first_seen_customers = tidy.dropDuplicates(["customer_id"])

first_seen_customers.count()

300

## Putting It Together

A typical cleaning step combines everything above into one function that
takes and returns a DataFrame, so it can be tested and reused:

In [30]:
from pyspark.sql import DataFrame


def clean_transactions(df: DataFrame) -> DataFrame:
    return (
        df.withColumnsRenamed(
            {
                "transactionID": "transaction_id",
                "customerID": "customer_id",
                "franchiseID": "franchise_id",
                "unitPrice": "unit_price",
                "totalPrice": "total_price",
                "paymentMethod": "payment_method",
            }
        )
        .withColumns(
            {
                "sale_date": F.to_date("dateTime"),
                "sale_hour": F.hour("dateTime"),
                "order_size": F.when(F.col("quantity") >= 30, "large")
                .when(F.col("quantity") >= 10, "medium")
                .otherwise("small"),
            }
        )
        .filter((F.col("quantity") > 0) & (F.col("total_price") > 0))
        .dropDuplicates(["transaction_id"])
        .drop("cardNumber")
    )


clean = clean_transactions(transactions)
clean.show(5)

+--------------+-----------+------------+--------------------+--------------------+--------+----------+-----------+--------------+----------+---------+----------+
|transaction_id|customer_id|franchise_id|            dateTime|             product|quantity|unit_price|total_price|payment_method| sale_date|sale_hour|order_size|
+--------------+-----------+------------+--------------------+--------------------+--------+----------+-----------+--------------+----------+---------+----------+
|       1002961|    2000253|     3000047|2024-05-14 12:17:...|  Golden Gate Ginger|       8|         3|         24|          amex|2024-05-14|       12|     small|
|       1003007|    2000226|     3000047|2024-05-10 23:10:...|Austin Almond Bis...|      36|         3|        108|    mastercard|2024-05-10|       23|     large|
|       1003017|    2000108|     3000047|2024-05-16 16:34:...|Austin Almond Bis...|      40|         3|        120|    mastercard|2024-05-16|       16|     large|
|       1003068|    20

Next, {doc}`Grouping and Aggregating Data <02_grouping_and_aggregating_data>`
turns these cleaned rows into summaries.

## Resources

- [PySpark functions reference (Apache Spark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
- [ANSI compliance in Spark SQL (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-ansi-compliance.html)
- [Datetime patterns (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html)
- [NULL semantics (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-null-semantics.html)